# VectorTrainer Update Benchmark

Measure the cost of one DQN optimizer update for the current vector trainer data path. This isolates replay sampling, CPU-to-GPU transfer, forward passes, backward pass, and optimizer step. It does not run environments and does not change production code.

## Setup

In [ ]:
# cell: setup

from pathlib import Path
import os
import platform
import subprocess
import sys
import time

if not Path("hpo/src").exists():
    if not Path("/content").exists():
        raise RuntimeError("Run this notebook from the rl_lab repository root or in Colab.")
    subprocess.run(["git", "clone", "https://github.com/miwehle/rl_lab.git"], check=True)
    os.chdir("rl_lab")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "hpo/requirements.txt"],
    check=True,
)

for _path in ("dqn/src", "hpo/src"):
    if _path not in sys.path:
        sys.path.insert(0, _path)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from dqn.model import DQN
from dqn.vector_training import VectorReplayMemory

def gpu_name():
    try:
        return subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            text=True,
        ).strip()
    except Exception:
        return "no nvidia-smi"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", gpu_name())
print("device:", device)

## Benchmark Config

In [ ]:
# cell: benchmark-config; requires: setup

SEED = 123
OBSERVATION_SHAPE = (10,)
N_OBSERVATIONS = 10
N_ACTIONS = 4
HIDDEN_SIZE = 128
REPLAY_MEMORY_CAPACITY = 76_754
BATCH_SIZES = [128, 256, 512, 1024, 2048, 4096]
WARMUP_UPDATES = 50
MEASURE_UPDATES = 300
LEARNING_RATE = 0.0006229370728793535
GAMMA = 0.995
TAU = 0.002

torch.manual_seed(SEED)
np.random.seed(SEED)

print("replay_memory_capacity:", REPLAY_MEMORY_CAPACITY)
print("batch_sizes:", BATCH_SIZES)
print("warmup_updates:", WARMUP_UPDATES)
print("measure_updates:", MEASURE_UPDATES)

## Benchmark Helpers

In [ ]:
# cell: benchmark-helpers; requires: benchmark-config

def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def now():
    cuda_sync()
    return time.perf_counter()

def make_filled_memory(seed):
    rng = np.random.default_rng(seed)
    memory = VectorReplayMemory(REPLAY_MEMORY_CAPACITY, OBSERVATION_SHAPE, seed=seed)
    memory.states[:] = rng.normal(size=(REPLAY_MEMORY_CAPACITY, *OBSERVATION_SHAPE)).astype(np.float32)
    memory.next_states[:] = rng.normal(size=(REPLAY_MEMORY_CAPACITY, *OBSERVATION_SHAPE)).astype(np.float32)
    memory.actions[:] = rng.integers(N_ACTIONS, size=REPLAY_MEMORY_CAPACITY, dtype=np.int64)
    memory.rewards[:] = rng.normal(size=REPLAY_MEMORY_CAPACITY).astype(np.float32)
    memory.terminated[:] = rng.random(REPLAY_MEMORY_CAPACITY) < 0.05
    memory.position = 0
    memory.size = REPLAY_MEMORY_CAPACITY
    return memory

def make_training_objects():
    q_net = DQN(N_OBSERVATIONS, N_ACTIONS, HIDDEN_SIZE).to(device)
    target_net = DQN(N_OBSERVATIONS, N_ACTIONS, HIDDEN_SIZE).to(device)
    target_net.load_state_dict(q_net.state_dict())
    optimizer = optim.AdamW(q_net.parameters(), lr=LEARNING_RATE, weight_decay=0.01, amsgrad=True)
    return q_net, target_net, optimizer

def soft_target_update(q_net, target_net):
    with torch.no_grad():
        for target_param, policy_param in zip(target_net.parameters(), q_net.parameters(), strict=True):
            target_param.lerp_(policy_param, TAU)

def update_once(memory, q_net, target_net, optimizer, batch_size):
    t0 = now()
    batch = memory.sample(batch_size, device)
    t1 = now()

    states = batch.states.flatten(start_dim=1)
    next_states = batch.next_states.flatten(start_dim=1)
    q_values = q_net(states).gather(1, batch.actions)
    t2 = now()

    with torch.no_grad():
        next_q_values = target_net(next_states).max(1).values
        next_q_values[batch.terminated] = 0.0
        td_targets = batch.rewards + GAMMA * next_q_values
    t3 = now()

    loss = nn.functional.smooth_l1_loss(q_values.squeeze(1), td_targets)
    optimizer.zero_grad(set_to_none=True)
    t4 = now()
    loss.backward()
    torch.nn.utils.clip_grad_value_(q_net.parameters(), 100)
    t5 = now()
    optimizer.step()
    soft_target_update(q_net, target_net)
    t6 = now()

    return {
        "sample_transfer_ms": (t1 - t0) * 1000,
        "q_forward_ms": (t2 - t1) * 1000,
        "target_forward_ms": (t3 - t2) * 1000,
        "loss_zero_grad_ms": (t4 - t3) * 1000,
        "backward_clip_ms": (t5 - t4) * 1000,
        "optimizer_soft_update_ms": (t6 - t5) * 1000,
        "total_ms": (t6 - t0) * 1000,
    }

def measure_batch_size(batch_size):
    memory = make_filled_memory(SEED + batch_size)
    q_net, target_net, optimizer = make_training_objects()

    for _ in range(WARMUP_UPDATES):
        update_once(memory, q_net, target_net, optimizer, batch_size)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    rows = []
    for _ in range(MEASURE_UPDATES):
        rows.append(update_once(memory, q_net, target_net, optimizer, batch_size))

    df = pd.DataFrame(rows)
    means = df.mean().to_dict()
    means["batch_size"] = batch_size
    means["updates_per_second"] = 1000 / means["total_ms"]
    means["samples_per_second"] = batch_size * means["updates_per_second"]
    means["peak_cuda_memory_mb"] = (
        torch.cuda.max_memory_allocated() / 1024 / 1024 if torch.cuda.is_available() else None
    )
    return means

## Run Benchmark

In [ ]:
# cell: run-benchmark; requires: benchmark-helpers

rows = []
for batch_size in BATCH_SIZES:
    row = measure_batch_size(batch_size)
    rows.append(row)
    print(
        f"batch={batch_size:4d} "
        f"total={row['total_ms']:7.3f} ms "
        f"sample+transfer={row['sample_transfer_ms']:7.3f} ms "
        f"q_forward={row['q_forward_ms']:7.3f} ms "
        f"target_forward={row['target_forward_ms']:7.3f} ms "
        f"backward+clip={row['backward_clip_ms']:7.3f} ms "
        f"opt+target={row['optimizer_soft_update_ms']:7.3f} ms "
        f"updates/s={row['updates_per_second']:7.1f}"
    )

## Results

In [ ]:
# cell: results; requires: run-benchmark

results = pd.DataFrame(rows)
ordered_cols = [
    "batch_size",
    "total_ms",
    "sample_transfer_ms",
    "q_forward_ms",
    "target_forward_ms",
    "loss_zero_grad_ms",
    "backward_clip_ms",
    "optimizer_soft_update_ms",
    "updates_per_second",
    "samples_per_second",
    "peak_cuda_memory_mb",
]
display(results[ordered_cols])

share_cols = [
    "sample_transfer_ms",
    "q_forward_ms",
    "target_forward_ms",
    "loss_zero_grad_ms",
    "backward_clip_ms",
    "optimizer_soft_update_ms",
]
shares = results[["batch_size", *share_cols]].copy()
for col in share_cols:
    shares[col] = shares[col] / results["total_ms"]
display(shares)

try:
    ax = results.plot(
        x="batch_size",
        y=["sample_transfer_ms", "q_forward_ms", "target_forward_ms", "backward_clip_ms", "optimizer_soft_update_ms"],
        kind="bar",
        stacked=True,
        figsize=(10, 5),
        title="Mean optimizer update time by batch size",
    )
    ax.set_ylabel("ms per update")
except Exception as exc:
    print("plot skipped:", exc)

## Reading the Result

If `sample_transfer_ms` is small compared with `backward_clip_ms` and `optimizer_soft_update_ms`, GPU replay is unlikely to be the first KISS performance lever. If `sample_transfer_ms` is large, a GPU replay buffer or pinned-memory transfer path becomes more interesting. If larger batches increase `samples_per_second` strongly, batch size and update strategy may be the more natural next lever.